In [1]:
%%capture
import os
import sys


!{sys.executable} -m pip install pip3-autoremove
!{sys.executable} -m pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!{sys.executable} -m pip install unsloth
!{sys.executable} -m pip install transformers==4.56.2
!{sys.executable} -m pip install --no-deps trl==0.22.2
!{sys.executable} -m pip install nltk rouge_score absl-py
!{sys.executable} -m pip install bert_score
!{sys.executable} -m pip install scikit-learn
!pip install wandb
!pip install evaluate

In [2]:
import json
import datasets
from datasets import Dataset
import random

data_list = []
with open("/kaggle/input/datasets/vietanhng/adsssss/Test.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        data_list.append(data)


dataset = Dataset.from_list(data_list) 


In [3]:
dataset

Dataset({
    features: ['answer', 'question'],
    num_rows: 934
})

In [4]:
dataset[0]

{'answer': 'Chào bạn,\nHiện nay tình trạng thoái hóa xương khớp rất phổ biến ở nhân viên văn phòng do ngồi một chỗ , không vận động nhiều, hay làm việc trước máy tính, ít rèn luyện cơ thể và thường xuyên bị nặng ở những người làm việc lao động chân tay, những người cao tuổi.\nBạn nên hạn chế ngồi lâu, nên tập thể dục giữa giờ các bài tập cổ, vai , gáy có thể tập tại chỗ từ 15-20 phút để cải thiện tình hình.\nNgoài ra nên tập thể dục vào sáng và tối các môn thể thao tăng cường vận động cơ xương khớp như chạy , đi bộ, tennis, cầu lông...\nNếu tình trạng kéo dài và không đỡ bạn nên sớm đi khám để bác sỹ có hướng điều trị cụ thể cho bạn.\nThân mến!',
 'question': "Xin chào bác sĩ,\n\nGần đây cháu hay bị đau đầu khi làm việc nhiều, vùng bị đau là vùng sau gáy bên phải.\nThông thường làm việc tập trung liên tục khoảng 2-3h thì sẽ đau và không làm việc tiếp đc. Nếu nghỉ tầm 15-20' thì sẽ đỡ đau hơn.\n\nDo đặc thù công việc cháu chưa đi khám đc. Mong đc bác sĩ tư vấn ạ"}

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "NgVietAnh41/Qwen3-4b-it-VietMedQA-final-DPO"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left" #


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    # load_in_4bit=True,
    torch_dtype=torch.float16,
)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [6]:
test_dataset = dataset
test_dataset = [row for row in dataset]


In [7]:
test_dataset[:3]

[{'answer': 'Chào bạn,\nHiện nay tình trạng thoái hóa xương khớp rất phổ biến ở nhân viên văn phòng do ngồi một chỗ , không vận động nhiều, hay làm việc trước máy tính, ít rèn luyện cơ thể và thường xuyên bị nặng ở những người làm việc lao động chân tay, những người cao tuổi.\nBạn nên hạn chế ngồi lâu, nên tập thể dục giữa giờ các bài tập cổ, vai , gáy có thể tập tại chỗ từ 15-20 phút để cải thiện tình hình.\nNgoài ra nên tập thể dục vào sáng và tối các môn thể thao tăng cường vận động cơ xương khớp như chạy , đi bộ, tennis, cầu lông...\nNếu tình trạng kéo dài và không đỡ bạn nên sớm đi khám để bác sỹ có hướng điều trị cụ thể cho bạn.\nThân mến!',
  'question': "Xin chào bác sĩ,\n\nGần đây cháu hay bị đau đầu khi làm việc nhiều, vùng bị đau là vùng sau gáy bên phải.\nThông thường làm việc tập trung liên tục khoảng 2-3h thì sẽ đau và không làm việc tiếp đc. Nếu nghỉ tầm 15-20' thì sẽ đỡ đau hơn.\n\nDo đặc thù công việc cháu chưa đi khám đc. Mong đc bác sĩ tư vấn ạ"},
 {'answer': 'Chào b

In [8]:
import json
from tqdm import tqdm 


output_file = "eval_dataset.jsonl"
batch_size = 8

print(f"Bắt đầu xử lý {len(test_dataset)} mẫu")

for i in tqdm(range(0, len(test_dataset), batch_size)):
    batch_data = test_dataset[i : i + batch_size]
    raw_prompts = [item["question"] for item in batch_data]
    formatted_prompts = [
        tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
        for p in raw_prompts
    ]
    inputs = tokenizer(formatted_prompts, return_tensors="pt", padding=True, truncation=True).to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    with open(output_file, "a", encoding="utf-8") as f:
        for j, out in enumerate(outputs):
            input_len = inputs['input_ids'][j].shape[0]
            generated_tokens = out[input_len:]
            response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
            test_sample = {
                "question": batch_data[j]["question"],
                "reference": batch_data[j]["answer"], 
                "response": response         
            }
            # print(test_sample)
            f.write(json.dumps(test_sample, ensure_ascii=False) + "\n")

print("Hoàn tất! Dữ liệu đã sẵn sàng ở file:", output_file)

Bắt đầu xử lý 934 mẫu


100%|██████████| 117/117 [51:25<00:00, 26.37s/it]

Hoàn tất! Dữ liệu đã sẵn sàng ở file: eval_dataset.jsonl
